# Лабораторна робота №6: блокова обробка. Спрощений JPEG-подібний алгоритм

Реалізуємо **блокову обробку** \(8 \times 8\): **DCT** → **квантування** за таблицями як у JPEG → **де-квантування** → **IDCT** для каналів **Y**, **Cr**, **Cb** після перетворення **BGR → YCrCb**. Це спрощена модель JPEG **без** субдискретизації хроми та **без** ентропійного кодування (Хаффман / арифметичний), але з типовою структурою просторового стиснення.

## 1. Імпорт бібліотек

In [1]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from scipy.fftpack import dct, idct

## 2. Налаштування шляхів

Підтримка запуску з **кореня репозиторію** (`cwd/Lab_06/Lab_06.ipynb`) або з папки **`Lab_06`** (`cwd/Lab_06.ipynb`).

In [2]:
_cwd = Path.cwd().resolve()
NOTEBOOK_DIR = _cwd if (_cwd / "Lab_06.ipynb").is_file() else (_cwd / "Lab_06")
ROOT = NOTEBOOK_DIR.parent.resolve()
IMAGE_PATH = ROOT / "satir.jpg"
RESULTS_DIR = (NOTEBOOK_DIR / "results").resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("ROOT:", ROOT)
print("IMAGE_PATH:", IMAGE_PATH)
print("RESULTS_DIR:", RESULTS_DIR)

NOTEBOOK_DIR: C:\Users\feost\OneDrive\Документи\CNU Education\AISIP\Lab_06
ROOT: C:\Users\feost\OneDrive\Документи\CNU Education\AISIP
IMAGE_PATH: C:\Users\feost\OneDrive\Документи\CNU Education\AISIP\satir.jpg
RESULTS_DIR: C:\Users\feost\OneDrive\Документи\CNU Education\AISIP\Lab_06\results


## 3. Unicode-шляхи (Windows): читання кольору та запис

In [3]:
def imread_color_unicode(path: Path) -> np.ndarray:
    data = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Не вдалося прочитати (колір): {path}")
    return img


def imwrite_unicode(path: Path, image: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    ok, buf = cv2.imencode(path.suffix, image)
    if not ok:
        raise RuntimeError(f"Не вдалося закодувати: {path}")
    buf.tofile(str(path))


def bgr_to_rgb(image_bgr: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

## 4. Завантаження `satir.jpg` (BGR) та збереження оригіналу

Для **Matplotlib** використовуємо перетворення **BGR → RGB**.

In [4]:
image_bgr = imread_color_unicode(IMAGE_PATH)
imwrite_unicode(RESULTS_DIR / "original_color.png", image_bgr)

plt.figure(figsize=(5, 5))
plt.imshow(bgr_to_rgb(image_bgr))
plt.title("Оригінал (RGB для відображення)")
plt.axis("off")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "original_color_display.png", dpi=120, bbox_inches="tight")
plt.close()

h, w = image_bgr.shape[:2]
print(f"Розмір: {w}×{h}, каналів: {image_bgr.shape[2]}")

Розмір: 426×640, каналів: 3


## 5. Перетворення кольорового простору BGR → YCrCb

Яскравість **Y** та кольорорізницеві компоненти **Cr**, **Cb** зручні для стиснення: візуальна вага зосереджена переважно в **Y**, тож хрому можна квантувати сильніше (у повному JPEG ще й субдискретизують; тут — спрощено, повний розмір).

In [5]:
ycrcb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2YCrCb)
Y, Cr, Cb = cv2.split(ycrcb)

imwrite_unicode(RESULTS_DIR / "plane_Y.png", Y)
imwrite_unicode(RESULTS_DIR / "plane_Cr.png", Cr)
imwrite_unicode(RESULTS_DIR / "plane_Cb.png", Cb)

fig, ax = plt.subplots(1, 3, figsize=(10, 3.2))
ax[0].imshow(Y, cmap="gray")
ax[0].set_title("Y (яскравість)")
ax[1].imshow(Cr, cmap="gray")
ax[1].set_title("Cr")
ax[2].imshow(Cb, cmap="gray")
ax[2].set_title("Cb")
for a in ax:
    a.axis("off")
plt.suptitle("Канали YCrCb після розділення")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ycrcb_planes_display.png", dpi=130, bbox_inches="tight")
plt.close()

## 6. DCT / IDCT по блоках \(8 \times 8\) та квантування

Використовуємо **дві базові матриці** JPEG: для яскравості (**luma**) та для кольорорізниці (**chroma**). Параметр якості **Q** масштабує таблиці за типовою формулою ISO (як у `Lab_05`).

In [6]:
Q_LUMA = np.array(
    [
        [16, 11, 10, 16, 24, 40, 51, 61],
        [12, 12, 14, 19, 26, 58, 60, 55],
        [14, 13, 16, 24, 40, 57, 69, 56],
        [14, 17, 22, 29, 51, 87, 80, 62],
        [18, 22, 37, 56, 68, 109, 103, 77],
        [24, 35, 55, 64, 81, 104, 113, 92],
        [49, 64, 78, 87, 103, 121, 120, 101],
        [72, 92, 95, 98, 112, 100, 103, 99],
    ],
    dtype=np.float64,
)

Q_CHROMA = np.array(
    [
        [17, 18, 24, 47, 99, 99, 99, 99],
        [18, 21, 26, 66, 99, 99, 99, 99],
        [24, 26, 56, 99, 99, 99, 99, 99],
        [47, 66, 99, 99, 99, 99, 99, 99],
        [99, 99, 99, 99, 99, 99, 99, 99],
        [99, 99, 99, 99, 99, 99, 99, 99],
        [99, 99, 99, 99, 99, 99, 99, 99],
        [99, 99, 99, 99, 99, 99, 99, 99],
    ],
    dtype=np.float64,
)


def dct2(block: np.ndarray) -> np.ndarray:
    return dct(dct(block.T, norm="ortho").T, norm="ortho")


def idct2(block: np.ndarray) -> np.ndarray:
    return idct(idct(block.T, norm="ortho").T, norm="ortho")


def scale_quant_table(base: np.ndarray, quality: int) -> np.ndarray:
    q = int(np.clip(quality, 1, 100))
    if q < 50:
        scale = 5000.0 / q
    else:
        scale = 200.0 - 2.0 * q
    mat = np.floor((base * scale + 50.0) / 100.0)
    return np.maximum(mat, 1.0)


def compress_decompress_plane(plane: np.ndarray, Q_base: np.ndarray, quality: int) -> np.ndarray:
    Qm = scale_quant_table(Q_base, quality)
    hh, ww = plane.shape
    pad_h = (8 - hh % 8) % 8
    pad_w = (8 - ww % 8) % 8
    padded = np.pad(plane, ((0, pad_h), (0, pad_w)), mode="edge")
    H, W = padded.shape
    out = np.zeros_like(padded, dtype=np.float64)
    for i in range(0, H, 8):
        for j in range(0, W, 8):
            blk = padded[i : i + 8, j : j + 8].astype(np.float64) - 128.0
            c = dct2(blk)
            cq = np.round(c / Qm)
            r = idct2(cq * Qm) + 128.0
            out[i : i + 8, j : j + 8] = r
    out = np.clip(np.round(out), 0, 255).astype(np.uint8)
    return out[:hh, :ww]


def jpeg_like_reconstruct_bgr(Yp: np.ndarray, Crp: np.ndarray, Cbp: np.ndarray, quality: int) -> np.ndarray:
    Y2 = compress_decompress_plane(Yp, Q_LUMA, quality)
    Cr2 = compress_decompress_plane(Crp, Q_CHROMA, quality)
    Cb2 = compress_decompress_plane(Cbp, Q_CHROMA, quality)
    merged = cv2.merge([Y2, Cr2, Cb2])
    return cv2.cvtColor(merged, cv2.COLOR_YCrCb2BGR)


def psnr_bgr(orig: np.ndarray, recon: np.ndarray) -> tuple[float, float]:
    a = orig.astype(np.float64)
    b = recon.astype(np.float64)
    mse = float(np.mean((a - b) ** 2))
    if mse == 0.0:
        return mse, float("inf")
    return mse, float(10.0 * np.log10(255.0**2 / mse))

## 7. Відновлення кольорового зображення та оцінка якості

Будуємо відновлені **BGR** для кількох **Q**, обчислюємо **MSE** та **PSNR** по всіх каналах, зберігаємо PNG та графік залежності PSNR від **Q**.

In [7]:
qualities_save = [25, 50, 75]
qualities_curve = list(range(10, 100, 5))

saved: dict[int, np.ndarray] = {}
curve: list[tuple[int, float, float]] = []

for q in qualities_curve:
    rec = jpeg_like_reconstruct_bgr(Y, Cr, Cb, q)
    mse_v, psnr_v = psnr_bgr(image_bgr, rec)
    curve.append((q, mse_v, psnr_v))
    if q in qualities_save:
        saved[q] = rec
        imwrite_unicode(RESULTS_DIR / f"reconstructed_bgr_q{q:02d}.png", rec)

for q, mse_v, psnr_v in curve:
    if q in qualities_save or q % 20 == 0:
        print(f"Q={q:3d}  MSE={mse_v:10.2f}  PSNR={psnr_v:6.2f} дБ")

q_show = 50
diff = np.abs(image_bgr.astype(np.int16) - saved[q_show].astype(np.int16))
diff_vis = np.clip(diff * 3, 0, 255).astype(np.uint8)
imwrite_unicode(RESULTS_DIR / "difference_bgr_scaled.png", diff_vis)

Qs, MSEs, PSNRs = zip(*curve)
fig, ax1 = plt.subplots(figsize=(6.5, 4))
ax1.plot(Qs, PSNRs, "b-o", markersize=4)
ax1.set_xlabel("Якість Q")
ax1.set_ylabel("PSNR (BGR), дБ")
ax1.grid(True, alpha=0.3)
ax2 = ax1.twinx()
ax2.plot(Qs, MSEs, "r--s", markersize=3)
ax2.set_ylabel("MSE")
plt.title("Якість відновлення кольорового зображення")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "psnr_mse_vs_quality_bgr.png", dpi=130)
plt.close(fig)

figc, axes = plt.subplots(2, 2, figsize=(9, 9))
axes[0, 0].imshow(bgr_to_rgb(image_bgr))
axes[0, 0].set_title("Оригінал")
axes[0, 1].imshow(bgr_to_rgb(saved[25]))
axes[0, 1].set_title("Після обробки, Q=25")
axes[1, 0].imshow(bgr_to_rgb(saved[50]))
axes[1, 0].set_title("Q=50")
axes[1, 1].imshow(bgr_to_rgb(saved[75]))
axes[1, 1].set_title("Q=75")
for a in axes.ravel():
    a.axis("off")
plt.suptitle("Спрощений JPEG-подібний ланцюг (YCrCb + DCT + Q)")
plt.tight_layout()
figc.savefig(RESULTS_DIR / "comparison_color_reconstructions.png", dpi=125, bbox_inches="tight")
plt.close(figc)

Q= 20  MSE=     74.33  PSNR= 29.42 дБ
Q= 25  MSE=     59.66  PSNR= 30.37 дБ
Q= 40  MSE=     38.34  PSNR= 32.29 дБ
Q= 50  MSE=     33.28  PSNR= 32.91 дБ
Q= 60  MSE=     30.29  PSNR= 33.32 дБ
Q= 75  MSE=      5.21  PSNR= 40.96 дБ
Q= 80  MSE=      1.33  PSNR= 46.91 дБ


## 8. Перевірка створених файлів

In [8]:
expected = [
    "original_color.png",
    "original_color_display.png",
    "plane_Y.png",
    "plane_Cr.png",
    "plane_Cb.png",
    "ycrcb_planes_display.png",
    "reconstructed_bgr_q25.png",
    "reconstructed_bgr_q50.png",
    "reconstructed_bgr_q75.png",
    "difference_bgr_scaled.png",
    "psnr_mse_vs_quality_bgr.png",
    "comparison_color_reconstructions.png",
]
missing = [name for name in expected if not (RESULTS_DIR / name).is_file()]
if missing:
    raise FileNotFoundError("Відсутні файли: " + ", ".join(missing))
print("Усі результати лабораторної роботи №6 успішно створено.")

Усі результати лабораторної роботи №6 успішно створено.
